In [1]:
import pandas as pd
import numpy as np
import re
import os
import matplotlib as mpl
import json

from collections import Counter

import warnings
warnings.filterwarnings('ignore')

In [2]:
def count_list_of_list(list_of_lists):
    """
    Input: List of list of keywords
    Output: {keyword: count} dict
    """
    counter = Counter()

    # Allowed characters: letters, digits, spaces, and selected punctuation
    allowed_pattern = r"[a-z0-9\s\-/.,&'():]+"

    for sublist in list_of_lists:
        cleaned_items = set()
        for word in sublist:
            if not isinstance(word, str):
                continue
            w = word.strip().lower()

            if re.fullmatch(allowed_pattern, w):
                cleaned_items.add(w)

        counter.update(cleaned_items)

    return dict(counter)

def count_list(words):

    cleaned = [
        w.lower()
        for w in words
        if isinstance(w, str) and len(w.strip()) >= 1
    ]
    
    # Count the terms
    return dict(Counter(cleaned))

def count_unique_keywords(keywords):
    if not keywords:
        return {}

    # Case 1: List[str]
    if isinstance(keywords[0], str):
        return count_list(keywords)

    # Case 2: List[List[str]]
    if isinstance(keywords[0], (list, np.ndarray)):
        return count_list_of_list(keywords)

    # Otherwise raise error
    raise TypeError(
        f"Invalid keywords type: expected List[str] or List[List[str]], "
        f"got element of type {type(keywords[0])}"
    )


def counter_to_df(word_counts, store_name):
    df = pd.DataFrame(word_counts.items(), columns=['keyword', 'count'])
    df = df.sort_values(by='count', ascending=False, ignore_index=True)
    df['store_name'] = store_name
    return df

def item_list_to_df(store_keywords, store_name):
    word_count_dict = count_unique_keywords(store_keywords)
    res = counter_to_df(word_count_dict, store_name)
    return res
    
def join_keywords_in_list(list_of_lists, special_color="green", max_words=3):
    result = []
    for colors in list_of_lists:
        # Always include special_color if present
        has_special = special_color in colors

        # Take first max_words items, but ensure special_color is kept
        subset = colors[:max_words]
        if has_special and special_color not in subset:
            # Replace last element with special_color (preserve max length)
            if len(subset) == max_words:
                subset[-1] = special_color
            else:
                subset.append(special_color)

        # ✅ Only ensure special_color comes first, preserve original order otherwise
        subset = list(subset)
        subset.sort(key=lambda c: (c != special_color, c))

        # Join and wrap in list
        result.append([" ".join(subset)])
    return result


""" Add param to allow for combining of keywords """
def keyword_count_store_key_df(df, key, store_name, join_keywords = False):
    # Get [['cypress', 'fir'], ['magnolia']] 
    store_df = df[df.store_name == store_name]
    store_keywords =  store_df[key].values.tolist()

    # if flatten: store_keywords = ...
    if join_keywords:
        store_keywords = join_keywords_in_list(store_keywords)

    res = item_list_to_df(store_keywords, store_name)
    return res

In [3]:
def minmax_scale(data):
    """
    Scales a list or numpy array of numbers to the range [0, 100].
    Handles edge cases like constant values safely.
    """
    arr = np.array(data, dtype=float)
    min_val, max_val = np.min(arr), np.max(arr)

    # Avoid division by zero if all values are equal
    if min_val == max_val:
        return np.zeros_like(arr)

    scaled = (arr - min_val) / (max_val - min_val) * 100
    return scaled

def pivot_keyword_dfs(dfs: list[pd.DataFrame]) -> pd.DataFrame:
        
    if not dfs:
        return pd.DataFrame()

    # Combine all DataFrames
    merged_df = pd.concat(dfs, ignore_index=True)

    # Pivot table: keywords as rows, store_names as columns
    pivot_df = merged_df.pivot_table(
        index='keyword',
        columns='store_name',
        values='count',
        aggfunc='sum',  # if duplicates exist, sum counts
        fill_value=0
    )
    
    return pivot_df

def get_pct_stores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds a 'pct_stores' column showing the fraction of stores
    (columns) where the keyword has a count > 0. 
    Then sorts by pct_stores descending, then by row sum of counts descending.
    """
    pivot_df = df.copy()
    store_count = pivot_df.shape[1]
    
    # Compute pct_stores
    pivot_df['pct_stores'] = (pivot_df.gt(0).sum(axis=1) / store_count).round(2)
    
    # Compute row sum across numeric columns excluding pct_stores
    numeric_cols = pivot_df.select_dtypes(include='number').columns.tolist()
    sum_cols = [col for col in numeric_cols if col != 'pct_stores']
    
    item_totals = pivot_df[sum_cols].sum(axis=1).values
    pivot_df['score'] = minmax_scale(item_totals)/100
    
    # Sort first by pct_stores, then by row sum
    pivot_df = pivot_df.sort_values(by=['score', 'pct_stores'], ascending=False)
    pivot_df = pivot_df[[c for c in pivot_df.columns if c != 'pct_stores'] + ['pct_stores']]
    
    return pivot_df

In [4]:
def divide_df_by_store_sizes(df, store_sizes):
    print(df.columns)
    if len(df.columns) != len(store_sizes):
        raise ValueError("❌ Length of store_sizes must match number of DataFrame columns.")
    
    # Convert to Series for broadcasting
    divisors = pd.Series(store_sizes, index=df.columns)
    return df.divide(divisors, axis='columns')

def normalize_store_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Exclude non-numeric columns like 'pct_stores' if present
    numeric_df = df.select_dtypes(include='number')

    # Compute column-wise percentages
    pct_df = numeric_df.div(numeric_df.sum(axis=0), axis=1).round(2)

    # Retain non-numeric columns if any
    non_numeric = df.drop(columns=numeric_df.columns, errors='ignore')
    result = pd.concat([pct_df, non_numeric], axis=1)

    return result

def normalize_pivot_df(df, store_sizes, how = "store_size"):
    # print(df)
    if how == "store_size":
        return divide_df_by_store_sizes(df, store_sizes)
    else:
        return normalize_store_columns(df)

def generate_pivots(dfs, store_sizes, store_names = None, how = "store_size"):
    pivot_df = pivot_keyword_dfs(dfs)
    pivot_df = pivot_df[store_names]

    # EITHER BY STORE SIZE OF NORMALIZE
    pivot_df_pcts = normalize_pivot_df(pivot_df, store_sizes, how = how)

    pivot_df = pivot_df[store_names]
    pivot_df_pcts = pivot_df_pcts[store_names]
    
    return pivot_df, pivot_df_pcts

def aggregate_count_dfs(dfs, store_sizes, store_names = None, how = "store_size"):
    pivot_df, pivot_df_pcts = generate_pivots(dfs, store_sizes, store_names, how)
    # print(pivot_df)
    # MAKE SURE STORE COLUMNS MATCH

    count_df = get_pct_stores(pivot_df)
    pct_df = get_pct_stores(pivot_df_pcts)

    count_df = count_df[count_df.pct_stores > 0]
    pct_df = pct_df[pct_df.pct_stores > 0]

    return count_df, pct_df

def colorize_df(df: pd.DataFrame, base_cmap: str = "Greens", min_frac: float = 0.0, max_frac: float = 0.6):
    # Get original colormap
    cmap_orig = mpl.cm.get_cmap(base_cmap)
    
    # Slice colormap to soften extremes
    cmap_vals = cmap_orig(np.linspace(min_frac, max_frac, 256))
    # Remove alpha channel for pandas
    cmap_vals = cmap_vals[:, :-1]
    soft_cmap = mpl.colors.ListedColormap(cmap_vals)

    return (
        df.style
        .background_gradient(cmap=soft_cmap, vmin=0, vmax=100, axis=None)
        .format("{:.0f}%")
    )

def generate_color_map(dfs, store_sizes, store_names, how = "store_size"):
    count_pivot, pct_pivot = aggregate_count_dfs(dfs, store_sizes, store_names, how)
    return count_pivot, pct_pivot, colorize_df(pct_pivot*100)

In [5]:
def df_to_colortable(df, key, how = 'store_size', join_keywords = False):
    # Get df of keyword counts for each store
    store_names = df.store_name.unique()
    keyword_count_dfs = [keyword_count_store_key_df(df, key, store, join_keywords) for store in store_names]
    store_sizes = [len(df[df.store_name == store]) for store in store_names]
    print("store_sizes", store_sizes)

    # Pivot
    counts, pcts, color_table = generate_color_map(keyword_count_dfs, store_sizes, store_names, how)
    return counts, pcts, color_table

### Try!

##### Load df

In [6]:
df = pd.read_parquet("final_flower_america_df.parquet")

In [14]:
df[df['flower_type'].apply(lambda x: len(x) > 1)]

,Name,Description,Price,store_name,Store Type,Store Quality,Product Type,Product Link,Main Img Link,filename,plant_type,flower_type,flower_colors,leaf_type,product_type,container,how_many_flowers,name_normalized,flowers_in_name
83,Faux-Ever Large Cream White Fake Rose and Peon...,Make your wedding day stress-free when you cho...,$120.00,Aflora,Independent Website,A,All Artificial Flowers,https://www.afloral.com/collections/artificial...,https://www.afloral.com/cdn/shop/files/PDP_58_...,Image_84,flower,"[peony, rose]",[white],[],single stem,none,2,faux-ever large cream white fake rose and peon...,"[peony, rose]"
179,Pink Cream Fake Peony and Eucalyptus Flower Bo...,Embrace the versatility of premade artificial ...,$36.00,Aflora,Independent Website,A,All Artificial Flowers,https://www.afloral.com/collections/artificial...,https://www.afloral.com/cdn/shop/files/Afloral...,Image_180,flower,"[peony, hydrangea]","[pink, red, white]",[eucalyptus],single stem,none,2,pink cream fake peony and eucalyptus flower bo...,"[eucalyptus, peony]"
347,Coral Garden Arrangement,Elevate your home dcor with vibrant blooms. Sh...,$299,Balsam Hill,Independent Website,A,Spring Artificial Flowers & Floral Dcor,https://www.balsamhill.com/f/coral-garden-flor...,https://source.widen.net/content/zumk7pwyqz/we...,Image_3,flower,"[peony, dahlia, hydrangea, ranunculus]","[green, orange, peach]",[],arrangement,vase,4,coral garden arrangement,[]
348,Southern Charm Arrangement,Bring warmth and elegance to your home with th...,$299,Balsam Hill,Independent Website,A,Spring Artificial Flowers & Floral Dcor,https://www.balsamhill.com/f/southern-charm-fl...,https://source.widen.net/content/4uvhazrjlf/we...,Image_4,flower,"[rose, hydrangea]",[white],[],arrangement,pot,2,southern charm arrangement,[]
349,Outdoor Vivid Blooms Hanging Basket,Refresh your homes exterior with the Outdoor V...,$229,Balsam Hill,Independent Website,A,Spring Artificial Flowers & Floral Dcor,https://www.balsamhill.com/f/outdoor-vivid-blo...,https://source.widen.net/content/uba1nbhgfg/we...,Image_5,flower,"[ranunculus, zinnia, dahlia, daisy, hydrangea]","[cream, green, pink, red, white, yellow]",[ivy],arrangement,planter,5,outdoor vivid bloom hanging basket,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3474,"Mix Floral in Rectangle Vase 16""",Subscribe to our newsletters to receive 10% of...,$580.00,Winward,Independent Website,A,Luxurious Faux Flower Arrangements,https://www.winwardhome.com/products/mix-flora...,https://www.winwardhome.com/cdn/shop/files/WHD...,Image_302,flower,"[peony, rose, snapdragon]","[cream, peach, white]",[],arrangement,vase,3,mix floral in rectangle vase 16 '',[]
3476,"Caf au Lait Combo in Vase 25"" wide",Subscribe to our newsletters to receive 10% of...,$930.00,Winward,Independent Website,A,Luxurious Faux Flower Arrangements,https://www.winwardhome.com/products/cafe-au-l...,https://www.winwardhome.com/cdn/shop/files/WHD...,Image_304,flower,"[peony, rose, dahlia]","[pink, white]",[],arrangement,vase,3,caf au lait combo in vase 25 '' wide,[]
3478,"Mix Floral in Square Vase 9.5""",Subscribe to our newsletters to receive 10% of...,$800.00,Winward,Independent Website,A,Luxurious Faux Flower Arrangements,https://www.winwardhome.com/products/mix-flora...,https://www.winwardhome.com/cdn/shop/files/WHD...,Image_306,flower,"[rose, hydrangea, helleborus]","[cream, green, pink]",[],arrangement,vase,3,mix floral in square vase 9.5 '',[]
3484,"Tulip & Peony in Cylinder 9""",Subscribe to our newsletters to receive 10% of...,$500.00,Winward,Independent Website,A,Luxurious Faux Flower Arrangements,https://www.winwardhome.com/products/tulip-peo...,https://www.winwardhome.com/cdn/shop/files/SDP...,Image_312,flower,"[peony, zinnia, hydrangea]","[green, pink, yellow]",[],arrangement,vase,3,tulip & peony in cylinder 9 '',"[peony, tulip]"


In [10]:
df.plant_type.value_counts()

plant_type
flower    3032
leaves     345
berry       92
            42
Name: count, dtype: int64

In [12]:
df[df.plant_type == 'leaves']['Main Img Link'].values.tolist()

['https://www.afloral.com/cdn/shop/files/AA-N1-053_df023216-f416-436b-b840-b128c0533705.jpg',
 'https://static.athome.com/images/w_1000,h_1000,c_pad,f_auto,fl_lossy,q_auto/p/124354025/string-of-pearls-in-white-oak-pot-5.jpg',
 'https://static.athome.com/images/w_1000,h_1000,c_pad,f_auto,fl_lossy,q_auto/p/124329393/eucalyptus-water-garden-in-vase-20.jpg',
 'https://static.athome.com/images/w_1000,h_1000,c_pad,f_auto,fl_lossy,q_auto/p/124232376/aloe-plant-with-cement-planter-36.jpg',
 'https://static.athome.com/images/w_1000,h_1000,c_pad,f_auto,fl_lossy,q_auto/p/124222852/greenery-arrangement-with-gold-wire-frame-6.jpg',
 'https://static.athome.com/images/w_1000,h_1000,c_pad,f_auto,fl_lossy,q_auto/p/124354019/variegated-greenery-in-metallic-pot-10.jpg',
 'https://static.athome.com/images/w_1000,h_1000,c_pad,f_auto,fl_lossy,q_auto/p/124232376/aloe-plant-with-cement-planter-36.jpg',
 'https://static.athome.com/images/w_1000,h_1000,c_pad,f_auto,fl_lossy,q_auto/p/124313872/eucalyptus-plant-w

#### quick view

In [8]:
flowers = df[df.plant_type == 'flower']
flowers = df[df.how_many_flowers > 0]
arrangements = flowers[flowers.product_type == 'arrangement']
single_stems = flowers[flowers.product_type == 'single stem']

pd.DataFrame(flowers.product_type.value_counts(normalize = True)).round(2)

,proportion
product_type,
arrangement,0.68
single stem,0.29
wreath,0.02
other,0.01
garland,0.00
flowerhead,0.00
teardrop,0.00


#### One flower vs. multiple flowers

In [9]:
pd.DataFrame(flowers.how_many_flowers.value_counts(normalize = True)).round(2)

,proportion
how_many_flowers,
1,0.83
2,0.09
3,0.06
4,0.03
5,0.00
6,0.00
7,0.00


### DF collection

- Keyword usage needs to be consistent and professional. Best achieved if we provide a standard list of possible choices. Otherwise, it will be all over the place. For instance, GPT might call 1 style 2 different words...Definition of each word needs to be very clear.
- Need to make sure keywords aren't being used randomly...(might want to remove text generation...). Maybe use text generation ONLY for the material. 
- 

In [10]:
def filter_rows_by_keywords(df, col, searchwords):
    if not isinstance(searchwords, (list, tuple)):
        raise ValueError("searchwords must be a list or tuple of strings.")

    mask = df[col].apply(lambda lst: all(word in lst for word in searchwords))
    return df[mask]

In [11]:
# rustic = filter_rows_by_keywords(df, "style", ['rustic'])

#### Flower type

In [17]:
counts, pcts, color_table = df_to_colortable(df, "flower_type")
color_table

store_sizes [327, 18, 115, 53, 218, 351, 185, 79, 189, 116, 249, 616, 88, 166, 118, 141, 52, 90, 340]
Index(['Aflora', 'At Home', 'Balsam Hill', 'Bloomist', 'Diane James Home',
       'Distinctive Designs', 'Emilio Robba', 'Faux Flower Company',
       'Flowerbx', 'Horchow', 'Nearly Natural', 'New Growth Designs',
       'Perigold', 'Pottery Barn', 'Prestige Botanicals', 'Real Touch Flowers',
       'Serene Spaces', 'Stilla', 'Winward'],
      dtype='object', name='store_name')


store_name,Aflora,At Home,Balsam Hill,Bloomist,Diane James Home,Distinctive Designs,Emilio Robba,Faux Flower Company,Flowerbx,Horchow,Nearly Natural,New Growth Designs,Perigold,Pottery Barn,Prestige Botanicals,Real Touch Flowers,Serene Spaces,Stilla,Winward,score,pct_stores
keyword,,,,,,,,,,,,,,,,,,,,,
hydrangea,18%,0%,23%,15%,27%,28%,5%,23%,1%,29%,22%,12%,15%,19%,14%,21%,2%,51%,18%,100%,95%
rose,17%,11%,13%,4%,22%,16%,14%,13%,26%,23%,9%,5%,15%,8%,17%,17%,10%,64%,18%,94%,100%
orchid,6%,0%,3%,0%,3%,27%,62%,3%,8%,24%,22%,6%,35%,3%,11%,8%,15%,13%,19%,79%,89%
peony,11%,0%,10%,2%,25%,8%,0%,14%,1%,12%,8%,7%,7%,7%,8%,1%,4%,20%,10%,45%,89%
ranunculus,5%,0%,5%,0%,14%,4%,0%,10%,0%,11%,3%,6%,5%,3%,5%,4%,2%,10%,7%,28%,79%
tulip,2%,0%,4%,0%,3%,6%,0%,10%,9%,6%,2%,6%,8%,1%,5%,13%,0%,0%,10%,25%,74%
dahlia,2%,0%,15%,0%,14%,1%,0%,0%,0%,5%,4%,1%,1%,2%,4%,0%,0%,16%,4%,20%,63%
magnolia,3%,0%,2%,2%,3%,2%,7%,5%,0%,4%,1%,1%,5%,2%,3%,0%,0%,0%,8%,14%,74%
calla lily,1%,0%,0%,0%,0%,1%,14%,1%,4%,3%,3%,2%,6%,0%,0%,6%,0%,1%,4%,14%,68%


In [18]:
counts.index.values.tolist()

['hydrangea',
 'rose',
 'orchid',
 'peony',
 'ranunculus',
 'tulip',
 'dahlia',
 'magnolia',
 'calla lily',
 'cherryblossom',
 'lily',
 'anemone',
 'daisy',
 'delphinium',
 'amaryllis',
 'lilac',
 'chrysanthemum',
 'carnation',
 'hellebore',
 'poppy',
 'protea',
 'anthurium',
 'allium',
 'poinsettia',
 'sunflower',
 'zinnia',
 'lavender',
 'gladiolus',
 'viburnum',
 'snapdragon',
 'hyacinth',
 'jasmine',
 'geranium',
 'narcissus',
 'iris',
 'bougainvillea',
 'pansy',
 'sweet pea',
 'bird of paradise',
 'gardenia',
 'lily of the valley',
 "baby's breath",
 'lisianthus',
 'camellia',
 'dogwood',
 'petunia',
 'bromeliad',
 'heliconia',
 'waxflower',
 'freesia',
 'grape hyacinth',
 'pincushion protea',
 'forsythia',
 'cosmos',
 'bells of ireland',
 'hibiscus',
 'amaranthus',
 'yarrow',
 'succulent',
 'helleborus',
 'mimosa',
 'astilbe',
 'flower',
 'primrose',
 'tuberose',
 'agapanthus',
 'cornflower',
 'gerbera',
 'stock',
 'wisteria',
 'artichoke',
 'azalea',
 'scabiosa',
 'african viole

In [13]:
counts, pcts, color_table = df_to_colortable(df, "flowers_in_name")
color_table

store_sizes [327, 18, 115, 53, 218, 351, 185, 79, 189, 116, 249, 616, 88, 166, 118, 141, 52, 90, 340]
Index(['Aflora', 'At Home', 'Balsam Hill', 'Bloomist', 'Diane James Home',
       'Distinctive Designs', 'Emilio Robba', 'Faux Flower Company',
       'Flowerbx', 'Horchow', 'Nearly Natural', 'New Growth Designs',
       'Perigold', 'Pottery Barn', 'Prestige Botanicals', 'Real Touch Flowers',
       'Serene Spaces', 'Stilla', 'Winward'],
      dtype='object', name='store_name')


store_name,Aflora,At Home,Balsam Hill,Bloomist,Diane James Home,Distinctive Designs,Emilio Robba,Faux Flower Company,Flowerbx,Horchow,Nearly Natural,New Growth Designs,Perigold,Pottery Barn,Prestige Botanicals,Real Touch Flowers,Serene Spaces,Stilla,Winward,score,pct_stores
keyword,,,,,,,,,,,,,,,,,,,,,
hydrangea,15%,6%,9%,9%,2%,25%,5%,10%,0%,16%,17%,12%,7%,14%,10%,20%,2%,3%,10%,100%,95%
rose,17%,0%,4%,6%,3%,9%,14%,6%,25%,16%,8%,7%,9%,8%,16%,17%,4%,8%,12%,98%,95%
orchid,6%,0%,1%,0%,0%,25%,22%,4%,8%,22%,16%,7%,23%,2%,11%,9%,15%,0%,16%,97%,84%
peony,11%,6%,4%,2%,3%,6%,0%,14%,1%,9%,8%,7%,7%,5%,7%,7%,4%,3%,8%,58%,95%
phalaenopsis,2%,0%,1%,0%,0%,4%,39%,0%,2%,2%,12%,2%,5%,1%,6%,4%,6%,0%,4%,47%,74%
lily,2%,6%,0%,0%,0%,2%,12%,3%,8%,4%,4%,4%,7%,1%,2%,9%,4%,0%,5%,37%,79%
tulip,2%,0%,2%,0%,0%,4%,0%,8%,7%,5%,2%,6%,6%,1%,6%,13%,0%,0%,9%,36%,68%
magnolia,3%,0%,2%,0%,2%,2%,8%,5%,0%,6%,1%,2%,5%,4%,3%,0%,0%,0%,12%,28%,68%
berry,1%,6%,0%,8%,0%,6%,0%,5%,1%,3%,6%,2%,0%,8%,0%,0%,2%,0%,1%,25%,63%


#### Flower combinations

1. Filter all rows where there's more than 1 flower
2. Look at the flowers!

In [76]:
multiple_flowers = flowers[flowers.how_many_flowers > 1]

In [77]:
counts, pcts, color_table = df_to_colortable(multiple_flowers, "flower_type", join_keywords = True)
color_table

store_sizes [2, 49, 86, 99, 10, 10, 4, 32, 34, 29, 10, 22, 1, 67, 60]
Index(['Aflora', 'Balsam Hill', 'Diane James Home', 'Distinctive Designs',
       'Emilio Robba', 'Faux Flower Company', 'Flowerbx', 'Horchow',
       'Nearly Natural', 'New Growth Designs', 'Perigold', 'Pottery Barn',
       'Serene Spaces', 'Stilla', 'Winward'],
      dtype='object', name='store_name')


store_name,Aflora,Balsam Hill,Diane James Home,Distinctive Designs,Emilio Robba,Faux Flower Company,Flowerbx,Horchow,Nearly Natural,New Growth Designs,Perigold,Pottery Barn,Serene Spaces,Stilla,Winward,score,pct_stores
keyword,,,,,,,,,,,,,,,,,
ranunculus rose,0%,0%,1%,0%,0%,0%,0%,3%,0%,0%,0%,5%,100%,1%,2%,100%,40%
hydrangea rose,0%,2%,6%,13%,0%,10%,0%,6%,6%,0%,10%,14%,0%,13%,15%,85%,67%
peony rose,50%,0%,7%,1%,0%,0%,0%,0%,0%,0%,0%,9%,0%,7%,2%,68%,40%
hydrangea peony,50%,0%,1%,3%,0%,0%,0%,0%,12%,0%,0%,0%,0%,1%,2%,61%,40%
hydrangea orchid,0%,0%,0%,5%,20%,0%,0%,3%,0%,3%,0%,0%,0%,3%,2%,32%,40%
hydrangea ranunculus,0%,0%,2%,1%,0%,20%,0%,6%,0%,3%,0%,0%,0%,0%,2%,30%,40%
hydrangea ranunculus rose,0%,0%,2%,1%,0%,0%,0%,6%,0%,0%,20%,0%,0%,1%,0%,27%,33%
peony rose tulip,0%,2%,1%,1%,0%,0%,25%,0%,0%,0%,0%,0%,0%,0%,2%,27%,33%
calla lily lily tulip,0%,0%,0%,0%,0%,0%,25%,0%,0%,0%,0%,0%,0%,0%,0%,22%,7%


#### Container

In [38]:
container_counts = pd.DataFrame(arrangements.container.value_counts(normalize = True).round(2))
container_counts

,proportion
container,
vase,0.70
pot,0.21
planter,0.07
none,0.01
basket,0.01
other,0.00


#### Product type 

Single stem vs arrangement

In [21]:
counts, pcts, color_table = df_to_colortable(df, "product_type")
color_table

store_sizes [327, 18, 115, 53, 218, 351, 185, 79, 189, 116, 249, 616, 166, 118, 141, 52, 90]
Index(['Aflora', 'At Home', 'Balsam Hill', 'Bloomist', 'Diane James Home',
       'Distinctive Designs', 'Emilio Robba', 'Faux Flower Company',
       'Flowerbx', 'Horchow', 'Nearly Natural', 'New Growth Designs',
       'Pottery Barn', 'Prestige Botanicals', 'Real Touch Flowers',
       'Serene Spaces', 'Stilla'],
      dtype='object', name='store_name')


store_name,Aflora,At Home,Balsam Hill,Bloomist,Diane James Home,Distinctive Designs,Emilio Robba,Faux Flower Company,Flowerbx,Horchow,Nearly Natural,New Growth Designs,Pottery Barn,Prestige Botanicals,Real Touch Flowers,Serene Spaces,Stilla,score,pct_stores
keyword,,,,,,,,,,,,,,,,,,,
arrangement,4%,44%,37%,17%,92%,99%,99%,97%,53%,91%,82%,49%,96%,2%,6%,79%,100%,100%,100%
single stem,91%,56%,1%,72%,8%,0%,1%,0%,28%,0%,17%,49%,4%,98%,91%,6%,0%,50%,82%
wreath,2%,0%,27%,9%,0%,1%,0%,1%,0%,9%,1%,0%,1%,0%,0%,2%,0%,5%,53%
other,1%,0%,18%,0%,0%,0%,0%,0%,19%,0%,0%,0%,0%,0%,0%,4%,0%,4%,29%
garland,2%,0%,13%,2%,0%,0%,0%,1%,0%,0%,0%,1%,0%,0%,0%,10%,0%,3%,41%
flowerhead,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,3%,0%,0%,0%,12%
teardrop,0%,0%,1%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,6%


In [22]:
product_type_distribution = pd.DataFrame(df.product_type.value_counts(normalize = True).round(2))
product_type_distribution

,proportion
product_type,
arrangement,0.64
single stem,0.29
wreath,0.03
other,0.02
garland,0.01
,0.00
flowerhead,0.00
teardrop,0.00


#### Colors

In [25]:
counts, pcts, color_table = df_to_colortable(df, "flower_colors")
color_table

store_sizes [327, 18, 115, 53, 218, 351, 185, 79, 189, 116, 249, 616, 88, 166, 118, 141, 52, 90, 340]
Index(['Aflora', 'At Home', 'Balsam Hill', 'Bloomist', 'Diane James Home',
       'Distinctive Designs', 'Emilio Robba', 'Faux Flower Company',
       'Flowerbx', 'Horchow', 'Nearly Natural', 'New Growth Designs',
       'Perigold', 'Pottery Barn', 'Prestige Botanicals', 'Real Touch Flowers',
       'Serene Spaces', 'Stilla', 'Winward'],
      dtype='object', name='store_name')


store_name,Aflora,At Home,Balsam Hill,Bloomist,Diane James Home,Distinctive Designs,Emilio Robba,Faux Flower Company,Flowerbx,Horchow,Nearly Natural,New Growth Designs,Perigold,Pottery Barn,Prestige Botanicals,Real Touch Flowers,Serene Spaces,Stilla,Winward,score,pct_stores
keyword,,,,,,,,,,,,,,,,,,,,,
white,37%,11%,40%,15%,47%,19%,44%,54%,20%,46%,32%,37%,56%,28%,43%,27%,21%,62%,52%,100%,100%
pink,21%,11%,31%,9%,31%,15%,22%,25%,31%,23%,20%,18%,20%,17%,28%,37%,15%,28%,26%,62%,100%
yellow,8%,0%,21%,0%,9%,25%,6%,11%,8%,17%,18%,8%,8%,6%,6%,14%,2%,14%,13%,28%,89%
green,4%,0%,6%,2%,11%,16%,10%,9%,4%,14%,10%,9%,6%,7%,7%,4%,0%,23%,11%,22%,89%
red,8%,0%,5%,0%,7%,21%,12%,0%,13%,13%,19%,7%,2%,6%,8%,3%,2%,9%,6%,20%,84%
purple,7%,0%,15%,0%,7%,9%,5%,3%,12%,8%,20%,7%,10%,5%,4%,14%,0%,9%,4%,20%,84%
cream,7%,0%,7%,11%,10%,9%,3%,3%,1%,11%,6%,1%,7%,6%,3%,1%,0%,22%,7%,17%,89%
orange,3%,0%,4%,4%,5%,14%,2%,0%,6%,9%,7%,3%,10%,5%,3%,4%,0%,9%,7%,14%,84%
peach,4%,0%,3%,0%,11%,5%,1%,1%,2%,8%,0%,0%,6%,3%,4%,4%,0%,16%,6%,11%,84%


In [26]:
# counts, pcts, color_table = df_to_colortable(df, "flower_colors", join_keywords = True)
# color_table

In [27]:
counts.index.values.tolist()

['white',
 'pink',
 'yellow',
 'red',
 'green',
 'purple',
 'orange',
 'cream',
 'blue',
 'peach',
 'light green',
 'light blue',
 'light pink',
 'magenta',
 'brown',
 'light purple',
 'coral',
 'dark red',
 'tan',
 'pale pink',
 'beige',
 'dark pink',
 'pale yellow',
 'gold',
 'bright pink',
 'burgundy',
 'dark purple',
 'grey',
 'lavender',
 'violet']

#### Flower combination

In [51]:
def filter_rows_by_all_keywords(df, col, keywords, all_keywords = False):
    if not keywords or len(keywords) == 0:
        return df
    
    def contains_all_keywords(cell):
        # Handle NaN or None
        if cell is None:
            return False

        # Handle numpy arrays, lists, tuples, sets uniformly
        if isinstance(cell, (np.ndarray, list, tuple, set)):
            cell_iterable = cell
        else:
            # Try to detect NaN scalars
            if pd.isna(cell):
                return False
            cell_iterable = [cell]

        # Lowercase all items for case-insensitive comparison
        cell_strs = [str(item).lower() for item in cell_iterable]

        # Check if every keyword appears in any of the cell strings
        if all_keywords:
            return all(
                any(keyword.lower() in s for s in cell_strs)
                for keyword in keywords
            )
        else:
            return any(
                any(keyword.lower() in s for s in cell_strs)
                for keyword in keywords
            )

    mask = df[col].apply(contains_all_keywords)
    return df[mask]

In [54]:
hydrangea = filter_rows_by_all_keywords(flowers, 'flower_type', ['hydrangea'])

In [55]:
counts, pcts, color_table = df_to_colortable(hydrangea, "flower_type")
color_table

store_sizes [58, 26, 8, 58, 99, 9, 18, 1, 34, 54, 74, 13, 31, 16, 29, 1, 46, 62]
Index(['Aflora', 'Balsam Hill', 'Bloomist', 'Diane James Home',
       'Distinctive Designs', 'Emilio Robba', 'Faux Flower Company',
       'Flowerbx', 'Horchow', 'Nearly Natural', 'New Growth Designs',
       'Perigold', 'Pottery Barn', 'Prestige Botanicals', 'Real Touch Flowers',
       'Serene Spaces', 'Stilla', 'Winward'],
      dtype='object', name='store_name')


store_name,Aflora,Balsam Hill,Bloomist,Diane James Home,Distinctive Designs,Emilio Robba,Faux Flower Company,Flowerbx,Horchow,Nearly Natural,New Growth Designs,Perigold,Pottery Barn,Prestige Botanicals,Real Touch Flowers,Serene Spaces,Stilla,Winward,score,pct_stores
keyword,,,,,,,,,,,,,,,,,,,,
hydrangea,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%
rose,0%,19%,0%,31%,28%,11%,17%,0%,38%,4%,0%,31%,23%,0%,0%,0%,57%,39%,16%,61%
ranunculus,0%,15%,0%,17%,4%,0%,22%,0%,24%,7%,4%,15%,0%,0%,0%,0%,9%,16%,7%,56%
orchid,0%,4%,0%,5%,14%,78%,0%,0%,6%,0%,1%,8%,0%,0%,0%,0%,9%,5%,7%,50%
peony,2%,12%,0%,14%,10%,0%,0%,0%,12%,19%,1%,8%,6%,0%,0%,0%,20%,10%,6%,61%
dahlia,0%,23%,0%,12%,3%,0%,0%,0%,6%,7%,0%,0%,10%,0%,0%,0%,13%,6%,4%,44%
tulip,0%,8%,0%,3%,8%,0%,6%,0%,15%,0%,1%,15%,0%,0%,0%,0%,0%,10%,4%,44%
calla lily,0%,0%,0%,2%,1%,44%,0%,0%,6%,6%,1%,0%,0%,0%,0%,0%,0%,3%,3%,39%
lily,0%,0%,0%,3%,4%,22%,0%,0%,6%,2%,0%,0%,0%,0%,0%,0%,0%,3%,2%,33%
